This notebook extends `brain_age_pipeline.ipynb` and requires the following variables
- `fc_matrices_child`, `y_child_age`, `y_pred_all`, `gap_all`, `atlas`

> Korean language prompts and report outputs are intentional.

> Reports target Korean-speaking caregivers

In [ ]:
# Network ROI Definition and Strength Computation
import numpy as np

networks = {
    'DMN'   : [24, 27, 29, 30, 20],
    '언어'  : [4, 5, 8, 9, 45],
    '시각주의': [21, 22, 31, 47],
}

def get_network_strength(fc_matrix, roi_indices):
    """Compute mean within-network connectivity strength."""
    sub = fc_matrix[np.ix_(roi_indices, roi_indices)]
    idx = np.triu_indices(len(roi_indices), k=1)
    return sub[idx].mean()

network_names = list(networks.keys())
network_strengths = np.zeros((len(fc_matrices_child), len(networks)))

for subj_idx, fc in enumerate(fc_matrices_child):
    for net_idx, (net_name, roi_idx) in enumerate(networks.items()):
        network_strengths[subj_idx, net_idx] = get_network_strength(fc, roi_idx)

network_mean = network_strengths.mean(axis=0)
network_std  = network_strengths.std(axis=0)
network_zscores = (network_strengths - network_mean) / network_std


In [ ]:
# Peer relative FC Analysis and ROI Profiling
from collections import defaultdict

roi_names = [label for label in atlas.labels if label != 'Background']
mean_fc = fc_matrices_child.mean(axis=0)  # (48, 48) global mean FC


def get_network_zscores_peer(subject_idx, y_child_age,
                              network_strengths, network_names,
                              peer_range=1.5):
    actual_age = y_child_age[subject_idx]
    peer_mask = (
        (np.abs(y_child_age - actual_age) <= peer_range) &
        (np.arange(len(y_child_age)) != subject_idx)
    )
    peer_strengths = network_strengths[peer_mask]
    subj_strengths = network_strengths[subject_idx]
    peer_mean = peer_strengths.mean(axis=0)
    peer_std  = peer_strengths.std(axis=0)
    peer_std  = np.where(peer_std == 0, 1e-8, peer_std)
    zscores   = (subj_strengths - peer_mean) / peer_std
    return {name: zscores[i] for i, name in enumerate(network_names)}, peer_mask.sum()


def get_top_deviations_peer(subject_idx, fc_matrices,
                             y_child_age, roi_names,
                             peer_range=1.5, top_n=5):
    actual_age = y_child_age[subject_idx]
    peer_mask = (
        (np.abs(y_child_age - actual_age) <= peer_range) &
        (np.arange(len(y_child_age)) != subject_idx)
    )
    peer_mean_fc = fc_matrices[peer_mask].mean(axis=0)
    dev = fc_matrices[subject_idx] - peer_mean_fc
    row_idx, col_idx = np.triu_indices(48, k=1)
    deviations = dev[row_idx, col_idx]
    abs_dev    = np.abs(deviations)
    sorted_idx = np.argsort(abs_dev)[::-1][:top_n]
    return [(roi_names[row_idx[i]], roi_names[col_idx[i]], deviations[i])
            for i in sorted_idx]


def summarize_deviations(top_devs, top_n_rois=3):
    roi_devs = defaultdict(list)
    for roi_a, roi_b, dev_val in top_devs:
        roi_devs[roi_a].append(dev_val)
        roi_devs[roi_b].append(dev_val)
    summary = [(roi, np.mean(devs)) for roi, devs in roi_devs.items()]
    summary.sort(key=lambda x: abs(x[1]), reverse=True)
    return [(roi, mean_dev, 'strong' if mean_dev > 0 else 'weak')
            for roi, mean_dev in summary[:top_n_rois]]


# ROI Korean mapping table
roi_korean_map = {
    0:  ("Frontal Pole",                              "전두극",           "고차 인지, 계획"),
    1:  ("Insular Cortex",                            "뇌섬엽",           "감정 인식, 내수용 감각"),
    2:  ("Superior Frontal Gyrus",                    "상전두이랑",        "작업 기억, 자기 인식"),
    3:  ("Middle Frontal Gyrus",                      "중전두이랑",        "인지 조절, 주의"),
    4:  ("Inferior Frontal Gyrus, pars triangularis", "하전두이랑 삼각부", "언어 이해, 브로카 영역"),
    5:  ("Inferior Frontal Gyrus, pars opercularis",  "하전두이랑 피개부", "언어 산출, 브로카 영역"),
    6:  ("Precentral Gyrus",                          "중심전이랑",        "운동 계획"),
    7:  ("Temporal Pole",                             "측두극",           "사회 인지, 감정 처리"),
    8:  ("Superior Temporal Gyrus, anterior",         "상측두이랑 전부",   "언어 이해, 청각"),
    9:  ("Superior Temporal Gyrus, posterior",        "상측두이랑 후부",   "언어 처리, 청각"),
    10: ("Middle Temporal Gyrus, anterior",           "중측두이랑 전부",   "의미 처리, 언어"),
    11: ("Middle Temporal Gyrus, posterior",          "중측두이랑 후부",   "언어, 의미 기억"),
    12: ("Middle Temporal Gyrus, temporooccipital",   "중측두이랑 측두후두부", "시각-언어 통합"),
    13: ("Inferior Temporal Gyrus, anterior",         "하측두이랑 전부",   "시각 객체 인식"),
    14: ("Inferior Temporal Gyrus, posterior",        "하측두이랑 후부",   "시각 객체 인식"),
    15: ("Inferior Temporal Gyrus, temporooccipital", "하측두이랑 측두후두부", "시각 처리"),
    16: ("Postcentral Gyrus",                         "중심후이랑",        "체감각 처리"),
    17: ("Superior Parietal Lobule",                  "상두정소엽",        "시공간 처리, 주의"),
    18: ("Supramarginal Gyrus, anterior",             "연상회 전부",       "언어, 음운 처리"),
    19: ("Supramarginal Gyrus, posterior",            "연상회 후부",       "언어, 읽기"),
    20: ("Angular Gyrus",                             "각이랑",           "언어, 수 처리, DMN"),
    21: ("Lateral Occipital Cortex, superior",        "외측후두피질 상부", "시각 처리"),
    22: ("Lateral Occipital Cortex, inferior",        "외측후두피질 하부", "시각 처리"),
    23: ("Intracalcarine Cortex",                     "칼카린구 피질",     "일차 시각 처리"),
    24: ("Frontal Medial Cortex",                     "내측전두피질",      "DMN, 자기참조"),
    25: ("Juxtapositional Lobule Cortex",             "보완운동피질",      "운동 계획, 순서"),
    26: ("Subcallosal Cortex",                        "뇌량하피질",        "감정 조절"),
    27: ("Paracingulate Gyrus",                       "방대상이랑",        "사회 인지, 갈등 처리"),
    28: ("Cingulate Gyrus, anterior",                 "대상이랑 전부",     "감정 조절, 주의"),
    29: ("Cingulate Gyrus, posterior",                "대상이랑 후부",     "DMN, 자기참조"),
    30: ("Precuneous Cortex",                         "쐐기앞소엽",        "DMN, 시공간 처리"),
    31: ("Cuneal Cortex",                             "쐐기소엽",         "시각 처리"),
    32: ("Frontal Orbital Cortex",                    "안와전두피질",      "감정, 의사결정"),
    33: ("Parahippocampal Gyrus, anterior",           "해마방이랑 전부",   "기억, 공간 처리"),
    34: ("Parahippocampal Gyrus, posterior",          "해마방이랑 후부",   "기억, 공간 처리"),
    35: ("Lingual Gyrus",                             "혀이랑",           "시각 처리, 읽기"),
    36: ("Temporal Fusiform Cortex, anterior",        "측두방추이랑 전부", "얼굴 인식, 사회 인지"),
    37: ("Temporal Fusiform Cortex, posterior",       "측두방추이랑 후부", "얼굴/사물 인식"),
    38: ("Temporal Occipital Fusiform Cortex",        "측두후두방추피질",  "시각 객체 인식"),
    39: ("Occipital Fusiform Gyrus",                  "후두방추이랑",      "시각 처리"),
    40: ("Frontal Opercular Cortex",                  "전두덮개피질",      "언어, 삼키기"),
    41: ("Central Opercular Cortex",                  "중심덮개피질",      "체감각, 언어"),
    42: ("Parietal Opercular Cortex",                 "두정덮개피질",      "체감각, 청각"),
    43: ("Planum Polare",                             "극평면",           "청각, 언어"),
    44: ("Heschl's Gyrus",                           "헤슐이랑",         "일차 청각 피질"),
    45: ("Planum Temporale",                          "측두평면",         "언어, 청각 처리"),
    46: ("Supracalcarine Cortex",                     "칼카린구상피질",    "시각 처리"),
    47: ("Occipital Pole",                            "후두극",           "일차 시각 처리"),
}


def get_roi_profile(subject_idx, fc_matrices, y_child_age,
                    roi_korean_map, peer_range=1.5, top_n=5):
    actual_age = y_child_age[subject_idx]
    peer_mask = (
        (np.abs(y_child_age - actual_age) <= peer_range) &
        (np.arange(len(y_child_age)) != subject_idx)
    )
    peer_mean_fc = fc_matrices[peer_mask].mean(axis=0)
    dev = fc_matrices[subject_idx] - peer_mean_fc
    row_idx, col_idx = np.triu_indices(48, k=1)
    deviations = dev[row_idx, col_idx]
    pos_idx = np.where(deviations > 0)[0]
    neg_idx = np.where(deviations < 0)[0]
    pos_sorted = pos_idx[np.argsort(deviations[pos_idx])[::-1]][:top_n]
    neg_sorted = neg_idx[np.argsort(deviations[neg_idx])[:top_n]]

    def make_pair(idx):
        r, c = row_idx[idx], col_idx[idx]
        return (roi_korean_map[r][1], roi_korean_map[r][2],
                roi_korean_map[c][1], roi_korean_map[c][2],
                deviations[idx])

    return [make_pair(i) for i in pos_sorted], [make_pair(i) for i in neg_sorted]


In [ ]:
# LLM Prompt Builder and Mistral API
# Reports target Korean-speaking caregivers

def build_prompt(actual_age, predicted_age, bag,
                 peer_rank, uncertainty, n_peers,
                 network_zscores, roi_summary,
                 peer_bag_mean, peer_bag_std,
                 strong_pairs, weak_pairs,
                rank_label, rank_detail):

    bag_low    = round(peer_bag_mean - peer_bag_std, 2)
    bag_high   = round(peer_bag_mean + peer_bag_std, 2)
    in_range   = bag_low <= bag <= bag_high
    all_strong = all(z > 0.5 for z in network_zscores.values())
    all_weak   = all(z < -0.5 for z in network_zscores.values())
    all_avg    = all(-0.5 <= z <= 0.5 for z in network_zscores.values())

    bag_months  = round(abs(bag) * 12)
    direction   = "앞선" if bag > 0 else "뒤처진"

    name_map = {
        'DMN': '사회인지 네트워크',
        '언어': '언어 네트워크',
        '시각주의': '시각 처리 네트워크'
    }

    network_text = ""
    for name, z in network_zscores.items():
        if z > 0.5:
            level = "또래보다 강함"
        elif z < -0.5:
            level = "또래보다 약함"
        else:
            level = "또래 평균 수준"
        display_name = name_map.get(name, name)
        network_text += f"  - {display_name}: {level}\n"

    roi_strong_text = ""
    for kr_a, func_a, kr_b, func_b, dev in strong_pairs:
        roi_strong_text += f"  - {func_a} 영역과 {func_b} 영역 간 연결 강함\n"

    roi_weak_text = ""
    for kr_a, func_a, kr_b, func_b, dev in weak_pairs:
        roi_weak_text += f"  - {func_a} 영역과 {func_b} 영역 간 연결 약함\n"

    strong = [name_map.get(n, n) for n, z in network_zscores.items() if z > 0.5]
    type_guide = ""

    lang_strong  = network_zscores.get('언어', 0) > 0.5
    vis_strong   = network_zscores.get('시각주의', 0) > 0.5
    soc_strong   = network_zscores.get('DMN', 0) > 0.5

    if lang_strong and vis_strong and soc_strong:
        type_guide = "균형 심화형"
        type_desc  = "언어, 시각, 사회인지 네트워크 모두 또래보다 강함"
    elif lang_strong and vis_strong:
        type_guide = "언어·시각형"
        type_desc  = "언어, 시각 네트워크가 또래보다 강함"
    elif lang_strong and soc_strong:
        type_guide = "언어·사회인지형"
        type_desc  = "언어, 사회인지 네트워크가 또래보다 강함"
    elif vis_strong and soc_strong:
        type_guide = "시각·사회인지형"
        type_desc  = "시각, 사회인지 네트워크가 또래보다 강함"
    elif lang_strong:
        type_guide = "언어 집중형"
        type_desc  = "언어 네트워크가 또래보다 강함"
    elif vis_strong:
        type_guide = "시각 집중형"
        type_desc  = "시각주의 네트워크가 또래보다 강함"
    elif soc_strong:
        type_guide = "사회인지 집중형"
        type_desc  = "사회인지 네트워크가 또래보다 강함"
    else:
        type_guide = "균형 발달형"
        type_desc  = "세 네트워크 모두 또래 평균 수준"

    system_prompt = f"""당신은 소아 뇌 발달 연구 보조 도구입니다.
fMRI 기능적 연결성 분석 결과를 바탕으로 아동 보호자가 읽을 해석 리포트를 작성합니다.

반드시 아래 규칙을 따르세요:
1. 모든 언어 형식: 한국어(영어 금지), 습니다체, 모든 문장 끝 온점 필수
2. 마크다운 기호 절대 사용 금지 (**, *, -, '' 등)
3. "|" "—" 기호 절대 사용 금지
4. 소아 뇌 발달 연구 데이터(118명 규모)로 표현, ds000228 같은 기술 명칭 사용 금지
5. 네트워크는 반드시 아래 이름으로만 표현할 것.
   사회인지 네트워크, 언어 네트워크, 시각 처리 네트워크
   DMN 약어 사용 금지.
6. 뇌 발달 격차는 개월 수로 표현 ({bag_months}개월 {direction})
7. 또래 내 위치는 반드시 "상위 X%" 형식으로 표현할 것.
   예) 상위 18%, 상위 79%, 상위 100%
        "하위" 표현 사용 금지.
8. 영어 단어 및 알파벳 절대 사용 금지. 로마자 포함 금지.
   모든 내용 반드시 한국어로만 작성할 것.
   예) subtle → 미묘한, strengths → 강점, complex → 복잡한
   뇌 영역명도 반드시 한글로만 표기할 것.
   예) Temporal Gyrus → 측두이랑, Frontal Cortex → 전두피질
9. 아래 3개 파트 구조로만 작성. 파트당 2~3문장 이내.

[파트 구조]
발달 수준 및 또래 비교
뇌 연결 패턴
발달 활동 제안

[발달 수준 및 또래 비교 파트 필수 포함 항목]
아래 항목을 반드시 모두 포함해서 작성할 것.

① 예측 뇌 나이와 실제 나이 격차를 개월 수로 표현하고
   또래 몇 명 중 상위 몇%인지 한 문장으로 자연스럽게 연결할 것.
   예) "예측 뇌 나이는 9.5세로 실제 나이보다 약 6개월 앞서 있으며,
       또래 34명 중 상위 18%에 해당합니다."

② 격차가 오차 범위(±{uncertainty:.2f}세) 안에 포함되는 경우
   앞 문장에 이어서 자연스럽게 연결할 것.
   별도 문장으로 나열하지 말 것.
   예) "다만 이 차이는 예측 오차 범위 안에 포함되어
       또래 평균과 실질적인 차이는 크지 않습니다."
   오차 범위 밖인 경우 생략.

③ 강함 또는 약함 네트워크가 있으면 한 문장으로만 언급.
   없으면 생략.
   예) "언어 네트워크 연결이 강한 반면 사회인지 네트워크는 발달 여지가 있습니다."
   상세 내용은 뇌 연결 패턴 파트에서 다루므로 이 파트에서는 간략하게만 언급할 것.
   
[BAG와 네트워크 연결 강도의 관계]
뇌 발달 격차와 네트워크 연결 강도는 서로 다른 측면을 측정합니다.
"두 지표가 독립적"이라는 표현을 리포트에 직접 쓰지 말 것.
아래 상황에 맞게 발달 수준 및 또래 비교 파트에서 자연스럽게 연결해서 서술할 것:

뇌 발달 격차 음수 + 우수한 연결 영역 존재:
예) "전반적인 뇌 발달은 또래보다 완만하게 진행 중이나,
    특정 영역의 연결은 또래 대비 견고하게 형성되어 있습니다."

뇌 발달 격차 양수 + 보완이 필요한 연결 영역 존재:
예) "전반적인 뇌 발달은 또래보다 앞서 있으나,
    일부 영역의 연결은 상대적으로 발달 여지가 있습니다."

10. 뇌 연결 패턴 파트 작성 기준:
    네트워크별 연결 강도 데이터만 기반으로 작성할 것.
    ROI 데이터는 참고하지 말 것.
    반드시 아래 두 섹션 구조로 작성할 것.
    "섹션 1 -" "섹션 2 -" 같은 표기 절대 금지.

    우수한 연결 영역:
    반드시 "또래보다 강함"으로 표시된 네트워크만 언급할 것.
    "또래 평균 수준" 네트워크는 절대 포함하지 말 것.
    강함으로 표시된 네트워크를 모두 언급할 것.
    해당 네트워크가 담당하는 핵심 기능을 중심으로 서술.
    아동의 실제 생활에서 어떤 강점으로 나타나는지 친절하게 설명.
    강함 네트워크가 없으면 "현재까지 분석된 데이터에서 뚜렷한
    우수 연결 영역이 확인되지 않았습니다."로 서술.

    보완이 필요한 영역:
    반드시 "또래보다 약함"으로 표시된 네트워크만 언급할 것.
    "또래 평균 수준" 네트워크는 절대 포함하지 말 것.
    약함으로 표시된 네트워크를 모두 언급할 것.
    해당 네트워크의 핵심 기능을 중심으로 서술.
    아동의 실제 생활에서 어떤 보완점으로 나타나는지 친절하게 설명.
    반드시 "또래 평균보다 낮은 연결 강도를 보입니다" 표현 포함.
    약함 네트워크가 없으면 "현재까지 분석된 데이터에서 뚜렷한
    보완 영역이 확인되지 않았습니다."로 서술.

    전문 용어보다 기능 중심 설명에 집중할 것.
    각 섹션 2~3문장 이내.

    우수한 연결 영역
    (내용)

    보완이 필요한 영역
    (내용)

11. 발달 활동 제안 기준:
    강점으로 약점을 보완하는 활동 우선 제안.
    언어 강함: 이야기 만들기, 토론 놀이 등 심화 언어 활동 권장.
    언어 약함: 그림책 함께 읽기, 매일 짧은 대화 늘리기 권장.
    시각 강함: 입체 퍼즐, 건축 블록 등 심화 시공간 활동 권장.
    시각 약함: 색칠하기, 관찰 놀이 권장.
    사회인지 강함: 역할극, 팀 프로젝트 등 심화 사회 활동 권장. 단 타인 시선을 과하게 의식하지 않는지 함께 살펴볼 것.
    사회인지 약함: 감정 카드 놀이, 소집단 협동 놀이 권장.
    전부 강함: 균형 잡힌 심화 활동 지속 권장.


[출력 형식 엄수]
    각 파트 제목은 반드시 아래 단어만 단독으로 한 줄에 쓸 것.
    앞뒤에 번호, 콜론, 특수문자 절대 금지.
    반드시 아래 순서대로 출력할 것:

        발달 수준 및 또래 비교
        우수한 연결 영역
        보완이 필요한 영역
        발달 활동 제안"""

    user_prompt = f"""아래 수치를 바탕으로 이 아동의 뇌 발달 리포트를 작성해주세요.

[기본 정보]
- 실제 나이: {actual_age:.1f}세
- 예측 뇌 나이: {predicted_age:.1f}세 (±{uncertainty:.2f}세)
- 뇌 발달 격차: 약 {bag_months}개월 {direction} (수치: {bag:+.2f}세)
- 또래 내 위치: {rank_label} (또래 {n_peers}명 기준)
- 또래 순위: {rank_detail} 수준

[네트워크별 연결 강도 (또래 대비)]
{network_text}

[또래 내 위치]
- 또래 평균 뇌 발달 격차 범위: {bag_low:+.2f}세 ~ {bag_high:+.2f}세
- 해당 아동: {bag:+.2f}세, 범위 {'안' if in_range else '밖'} {'상위권' if bag > 0 else '하위권'} 해당

[데이터 맥락]
- 소아 뇌 발달 연구 데이터 (118명 규모) 기반 분석
- 측정 조건: 동일한 영상 시청 중 fMRI 측정
- 비교 기준: 동일 조건 또래 아동 {n_peers}명"""

    return system_prompt, user_prompt


def generate_llm_report(system_prompt, user_prompt):
    import requests
    import json

    api_key = "YOUR_MISTRAL_API_KEY"
    url = "https://api.mistral.ai/v1/chat/completions"

    payload = {
        "model": "mistral-small-latest",
        "temperature": 0,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
    }

    response = requests.post(
        url,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        },
        data=json.dumps(payload)
    )

    result = response.json()
    return result["choices"][0]["message"]["content"]

In [ ]:
# Visualization in Korean
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import re
import textwrap

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False


def clean_text(text):
    text = re.sub(r'\*+', '', text)
    text = re.sub(r"''+", '', text)
    text = re.sub(r'^\s*[-•]\s', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{2,}', '\n', text)
    text = text.replace('|', '').replace('—', '')
    return text.strip()


def draw_summary_bar(fig, actual_age, predicted_age,
                     bag, rank_label, uncertainty):
    gap_direction = "또래 대비 성숙" if bag > 0 else "또래 대비 미성숙"
    metrics = [
        ("실제 나이",    f"{actual_age:.1f}세",                           "#4A90D9"),
        ("예측 뇌 나이", f"{predicted_age:.1f}세\n(±{uncertainty:.2f}세)", "#5BA85A"),
        ("뇌 발달 격차", f"{bag:+.2f}세  ({gap_direction})",
         "#E05C5C" if bag > 0 else "#5BA85A"),
        ("또래 내 위치", rank_label,                                       "#9B6DD6"),
    ]
    card_width = 0.22
    gap = 0.015
    total_width = 4 * card_width + 3 * gap
    start_x = (1 - total_width) / 2

    for i, (label, value, color) in enumerate(metrics):
        x = start_x + i * (card_width + gap)
        ax = fig.add_axes([x, 0.885, card_width, 0.075])
        ax.set_facecolor(color)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
        ax.text(0.5, 0.75, label,
                transform=ax.transAxes,
                fontsize=11, color='white',
                ha='center', va='center', alpha=0.9)
        ax.text(0.5, 0.28, value,
                transform=ax.transAxes,
                fontsize=13, color='white',
                fontweight='bold',
                ha='center', va='center')


def draw_radar(ax, zscores, network_names):
    label_map = {
        'DMN': '사회인지',
        '언어': '언어',
        '시각주의': '시각 처리'
    }
    display_names = [label_map.get(n, n) for n in network_names]

    n = len(network_names)
    angles = np.linspace(0, 2 * np.pi, n + 1)[:-1]
    values = [zscores[name] for name in network_names]
    values_closed = values + [values[0]]
    angles_closed = list(angles) + [angles[0]]

    ax.set_facecolor('#F8F9FA')
    ax.set_ylim(-2, 2)
    ax.set_yticks([-1, 0, 1])
    ax.set_yticklabels([])

    for r in [-1, 0, 1]:
        if r == 0:
            ax.plot(angles_closed, [r] * (n + 1),
                    color='#888888', linewidth=1.5,
                    linestyle='--', alpha=0.8)
        else:
            ax.plot(angles_closed, [r] * (n + 1),
                    color='#CCCCCC', linewidth=0.8,
                    linestyle='--', alpha=0.6)

    ax.fill(angles_closed, values_closed,
            alpha=0.25, color='#4A90D9')
    ax.plot(angles_closed, values_closed,
            color='#4A90D9', linewidth=2,
            marker='o', markersize=6)

    ax.set_xticks(angles)
    ax.set_xticklabels([]) 


    label_padding = 2.6  
    for angle, name in zip(angles, display_names):
        ax.text(angle, label_padding, name,
            ha='center', va='center',
            fontsize=10, color='#333333')
    ax.set_title('네트워크별 연결 강도',
                 fontsize=11, fontweight='bold',
                 pad=15, color='#333333')
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#4A90D9', linewidth=2, label='해당 아동'),
        Line2D([0], [0], color='#888888', linewidth=1.5,
                linestyle='--', label='또래 평균 (z=0)')
    ]
    ax.legend(handles=legend_elements,
                loc='upper right',
                bbox_to_anchor=(1.15, 1.1),
                fontsize=8)


def draw_text_panel(ax, report_text):
    ax.set_facecolor('#F8F9FA')
    ax.axis('off')

    report_text = clean_text(report_text)
    report_text = re.sub(r'종합 분석[:\s]*(.*)', r'종합 분석\n\1', report_text)

    part_titles = [
        '발달 수준 및 또래 비교',
        '우수한 연결 영역',
        '보완이 필요한 영역',
        '발달 활동 제안',
    ]

    lines = report_text.split('\n')
    blocks = {}
    current_title = None
    current_body  = []

    for line in lines:
        clean_line = line.replace('*', '').replace('#', '').strip()
        if clean_line == '뇌 연결 패턴':
            continue
        if clean_line == '종합 분석':
            continue
        found_title = None
        for title in part_titles:
            if title in clean_line and len(clean_line) < len(title) + 5:
                found_title = title
                break
        if found_title:
            if current_title:
                blocks[current_title] = '\n'.join(current_body).strip()
            current_title = found_title
            current_body  = []
        elif current_title:
            current_body.append(line)

    if current_title:
        blocks[current_title] = '\n'.join(current_body).strip()

    left_col = [
        ('발달 수준 및 또래 비교', blocks.get('발달 수준 및 또래 비교', '')),
        ('우수한 연결 영역',       blocks.get('우수한 연결 영역', '')),
    ]
    right_col = [
        ('보완이 필요한 영역', blocks.get('보완이 필요한 영역', '')),
        ('발달 활동 제안',    blocks.get('발달 활동 제안', '')),
    ]


    ax.text(0.01, 1.1, '뇌 발달 해석 리포트',
            transform=ax.transAxes,
            fontsize=17, fontweight='bold',
            color='#222222', va='top')

    ax.text(0.01, 0.97,
            '뇌 발달 격차: 또래 대비 전체적인 뇌 발달 수준',
            transform=ax.transAxes,
            fontsize=12, color='#666666', va='top')
    ax.text(0.01, 0.90,
            '네트워크 연결 강도: 특정 뇌 영역 간 연결 강도 '
            '(뇌 발달 격차와 독립적으로 산출)',
            transform=ax.transAxes,
            fontsize=12, color='#666666', va='top')
    ax.text(0.01, 0.83,
            '본 리포트는 동일 조건 또래와의 상대적 비교 결과이며 '
            '임상 진단을 대체하지 않습니다. '
            '전문가 상담과 함께 참고 자료로 활용하시기 바랍니다.',
            transform=ax.transAxes,
            fontsize=12, color='#666666', va='top',
            style='italic')

    def render_block(title, body, x_start, y, col_width):
        ax.text(x_start, y, title,
                transform=ax.transAxes,
                fontsize=15, fontweight='bold',
                color='#4A90D9', va='top')
        y -= 0.07
        if body:
            body = clean_text(body)
            wrapped = textwrap.fill(body, width=col_width)
            ax.text(x_start + 0.01, y, wrapped,
                    transform=ax.transAxes,
                    fontsize=12, color='#444444',
                    va='top', linespacing=1.7)
        return y - 0.30

    y_left = 0.72
    for title, body in left_col:
        if body:
            y_left = render_block(title, body, 0.01, y_left, 55)

    y_right = 0.72
    for title, body in right_col:
        if body:
            y_right = render_block(title, body, 0.52, y_right, 55)


def brain_age_report_v2(subject_idx):

    actual_age    = y_child_age[subject_idx]
    predicted_age = y_pred_all[subject_idx]
    bag           = predicted_age - actual_age

    age_bands = [(3, 6, 2.1), (6, 9, 1.4), (9, 13, 1.2)]
    uncertainty = next((mae for lo, hi, mae in age_bands
                        if lo <= actual_age < hi), 1.59)

    peer_mask = (
        (np.abs(y_child_age - actual_age) <= 1.5) &
        (np.arange(len(y_child_age)) != subject_idx)
    )
    peer_gaps     = gap_all[peer_mask]
    percentile    = (peer_gaps < bag).mean() * 100
    peer_rank     = 100 - percentile
    n_peers       = peer_mask.sum()
    peer_bag_mean = peer_gaps.mean()
    peer_bag_std  = peer_gaps.std()

    rank_num    = int((1 - peer_rank/100) * n_peers) + 1
    rank_label  = f"상위 {peer_rank:.0f}%"
    rank_detail = f"{rank_num}/{n_peers}명"

    zscores, _ = get_network_zscores_peer(
        subject_idx, y_child_age, network_strengths, network_names
    )
    top_devs = get_top_deviations_peer(
        subject_idx, fc_matrices_child, y_child_age, roi_names
    )
    roi_summary = summarize_deviations(top_devs)

    strong_pairs, weak_pairs = get_roi_profile(
        subject_idx, fc_matrices_child, y_child_age, roi_korean_map
    )

    system_prompt, user_prompt = build_prompt(
        actual_age, predicted_age, bag,
        peer_rank, uncertainty, n_peers,
        zscores, roi_summary,
        peer_bag_mean, peer_bag_std,
        strong_pairs, weak_pairs,
        rank_label, rank_detail 
    )
    report_text = generate_llm_report(system_prompt, user_prompt)

    fig = plt.figure(figsize=(18, 15), facecolor='white')

    fig.text(0.5, 0.995,
             f'아동 뇌 발달 리포트  (피험자 {subject_idx}번 아동)',
             ha='center', va='top',
             fontsize=17, fontweight='bold', color='#222222')

    draw_summary_bar(fig, actual_age, predicted_age,
                     bag, rank_label, uncertainty)

    gs = gridspec.GridSpec(
        2, 3,
        figure=fig,
        top=0.8, bottom=0.04,
        left=0.05, right=0.97,
        hspace=0.3, wspace=0.3,
        height_ratios=[1, 1.2]
    )

    ax_dist    = fig.add_subplot(gs[0, 0])
    ax_scatter = fig.add_subplot(gs[0, 1])
    ax_radar   = fig.add_subplot(gs[0, 2], projection='polar')
    ax_text    = fig.add_subplot(gs[1, :])

    def style_ax(ax, title):
        ax.set_facecolor('#F8F9FA')
        ax.set_title(title, fontsize=11,
                     fontweight='bold', color='#333333', pad=10)
        for spine in ax.spines.values():
            spine.set_color('#DDDDDD')
        ax.tick_params(colors='#555555', labelsize=9)

    style_ax(ax_dist, '뇌 발달 격차 또래 분포')
    ax_dist.hist(peer_gaps, bins=15,
                 color='#AED6F1', edgecolor='white', alpha=0.9)
    ax_dist.axvline(bag, color='#E05C5C', linewidth=2.5,
                    label=f'해당 아동  {bag:+.2f}세')
    ax_dist.axvline(0, color='#AAAAAA', linewidth=1,
                    linestyle='--', label='기준 (gap=0)')
    ax_dist.set_xlabel('뇌 발달 격차 (세)', fontsize=9, color='#555555')
    ax_dist.set_ylabel('또래 수', fontsize=9, color='#555555')
    ax_dist.legend(fontsize=8, framealpha=0.7)

    style_ax(ax_scatter, '뇌 나이 예측')
    ax_scatter.scatter(y_child_age, y_pred_all,
                       alpha=0.35, color='#AED6F1', s=35)
    ax_scatter.scatter(actual_age, predicted_age,
                       color='#E05C5C', s=150, zorder=5,
                       label=f'해당 아동 (idx={subject_idx})')
    ax_scatter.errorbar(actual_age, predicted_age,
                        yerr=uncertainty, fmt='none',
                        color='#E05C5C', capsize=6, linewidth=2)
    ax_scatter.plot([3, 13], [3, 13], color='#AAAAAA',
                    linewidth=1, linestyle='--')
    ax_scatter.set_xlabel('실제 나이 (세)', fontsize=9, color='#555555')
    ax_scatter.set_ylabel('예측 나이 (세)', fontsize=9, color='#555555')
    ax_scatter.legend(fontsize=8, framealpha=0.7)

    draw_radar(ax_radar, zscores, network_names)
 
    ax_text.set_facecolor('#F8F9FA')
    for spine in ax_text.spines.values():
        spine.set_color('#DDDDDD')
    draw_text_panel(ax_text, report_text)

    plt.show()

    print(f"실제 나이    : {actual_age:.1f}세")
    print(f"예측 나이    : {predicted_age:.1f}세  (±{uncertainty:.2f}세)")
    print(f"뇌 발달 격차 : {bag:+.2f}세  ({('또래 대비 성숙' if bag > 0 else '또래 대비 미성숙')})")
    print(f"또래 내 위치 : {rank_label}  ({rank_detail})")

brain_age_report_v2(subject_idx=88)